In [3]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm.notebook import tqdm

# 1. Klasör ve Yol Ayarları
folder_name = "data"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"'{folder_name}' klasörü oluşturuldu.")

file_path = os.path.join(folder_name, "06_hf_mega_qa_dataset.csv")

try:
    # 2. Veri Setini İndir
    print("Hugging Face üzerinden veri çekiliyor (495 bin satır)...")
    dataset = load_dataset("kayrab/patient-doctor-qa-tr-167732")
    df_raw = dataset['train'].to_pandas()
    
    print("Veri analizi ve sütun eşleme yapılıyor...")
    
    # 3. Doğru Sütun Eşlemesi (Explicit Mapping)
    # question_content = Hasta Sorusu
    # question_answer = Doktor Cevabı
    df_final = pd.DataFrame()
    df_final['source'] = ['HuggingFace_Mega_QA'] * len(df_raw)
    df_final['question'] = df_raw['question_content']
    df_final['answer'] = df_raw['question_answer']
    
    # 4. Veri Temizliği
    df_final = df_final.dropna(subset=['question', 'answer'])
    # Sadece dişe dokunur uzunluktaki soru-cevapları tut
    df_final = df_final[(df_final['question'].str.len() > 20) & (df_final['answer'].str.len() > 20)]
    
    # 5. Dosyaya Yazma (Mutlak Yol İle)
    print(f"Dosya kaydediliyor: {os.path.abspath(file_path)}")
    df_final.to_csv(file_path, index=False, encoding="utf-8-sig") # Excel'de düzgün görünmesi için utf-8-sig
    
    print("\n" + "="*50)
    print("✅ BAŞARIYLA TAMAMLANDI!")
    print(f"Toplam Kayıt Sayısı: {len(df_final)}")
    print(f"Dosya Konumu: {os.path.abspath(file_path)}")
    print("="*50)
    
    # Doğrulama Testi
    print("\nÖrnek Kontrol (İlk Kayıt):")
    print(f"SORU (Hasta): {df_final['question'].iloc[0][:150]}...")
    print(f"CEVAP (Doktor): {df_final['answer'].iloc[0][:150]}...")

except Exception as e:
    print(f"\n❌ KRİTİK HATA: {e}")

Hugging Face üzerinden veri çekiliyor (495 bin satır)...


Veri analizi ve sütun eşleme yapılıyor...
Dosya kaydediliyor: e:\LLM-FULLPARAMETER\data\06_hf_mega_qa_dataset.csv

✅ BAŞARIYLA TAMAMLANDI!
Toplam Kayıt Sayısı: 490079
Dosya Konumu: e:\LLM-FULLPARAMETER\data\06_hf_mega_qa_dataset.csv

Örnek Kontrol (İlk Kayıt):
SORU (Hasta): Babamın dişlerini yaptırmak istiyoruz. Yardımcı olabilir misiniz?...
CEVAP (Doktor): Tabii, size yardımcı olmaktan memnuniyet duyarım. Sorunuzu sorabilirsiniz....


In [4]:
import pandas as pd
import os

# Bir önceki adımda kaydettiğin dosyanın yolu
INPUT_FILE = "data/06_hf_mega_qa_dataset.csv" 
OUTPUT_FILE = "data/07_diabetes_expert_qa.csv"

# Diyabet ve Endokrinoloji radarına takılacak anahtar kelimeler
# Modelin uzmanlığını artırmak için bu listeye istediğin tıbbi terimi ekleyebilirsin
keywords = [
    'diyabet', 'şeker', 'insülin', 'hipoglisemi', 'hiperglisemi', 
    'hba1c', 'glikoz', 'endokrin', 'tip 1', 'tip 2', 'tokluk şekeri', 'açlık şekeri',
    'metabolizma', 'insülin direnci', 'diyabetik', 'endokrinoloji', 'matofin', 'diaformin'
]

print("Mega veri seti (490K) belleğe alınıyor. Lütfen bekleyin...")

try:
    df = pd.read_csv(INPUT_FILE)
    
    print(f"Toplam {len(df)} adet genel tıbbi diyalog Diyabet süzgecinden geçiriliyor...")
    
    # NaN (boş) verileri string'e çevirip küçük harf yapıyoruz
    # Soru VEYA Cevap içinde belirlediğimiz kelimelerden BİRİ bile geçiyorsa o satırı alıyoruz
    mask = df['question'].astype(str).str.lower().str.contains('|'.join(keywords)) | \
           df['answer'].astype(str).str.lower().str.contains('|'.join(keywords))
           
    df_filtered = df[mask]
    
    # Çektiğimiz niş veriyi yeni bir dosyaya kaydediyoruz
    df_filtered.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    
    print("\n" + "="*50)
    print("🎉 ALTIN MADENİ BULUNDU! (FİLTRELEME BAŞARILI)")
    print(f"Taranan Genel Veri: {len(df)}")
    print(f"Yakalanan Saf Diyabet/Endokrinolojisi Vakası: {len(df_filtered)}")
    print(f"Uzmanlık Veri Setin Şuraya Kaydedildi: {os.path.abspath(OUTPUT_FILE)}")
    print("="*50)
    
    if len(df_filtered) > 0:
        print("\n🩺 DİYABET FİLTRELİ ÖRNEK SORU:\n", df_filtered['question'].iloc[0][:200], "...\n")
        print("💊 DİYABET FİLTRELİ ÖRNEK CEVAP:\n", df_filtered['answer'].iloc[0][:200], "...")
        print("\nNot: Artık genel verilerin olduğu '06_hf_mega_qa_dataset.csv' dosyasını silebilirsin. Bize sadece bu yeni dosya lazım.")

except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

Mega veri seti (490K) belleğe alınıyor. Lütfen bekleyin...
Toplam 490079 adet genel tıbbi diyalog Diyabet süzgecinden geçiriliyor...

🎉 ALTIN MADENİ BULUNDU! (FİLTRELEME BAŞARILI)
Taranan Genel Veri: 490079
Yakalanan Saf Diyabet/Endokrinolojisi Vakası: 20810
Uzmanlık Veri Setin Şuraya Kaydedildi: e:\LLM-FULLPARAMETER\data\07_diabetes_expert_qa.csv

🩺 DİYABET FİLTRELİ ÖRNEK SORU:
 Merhaba, 9 Eylül tarihinde doğum yaptım, 3 günlükken şekeri düşen bebeğimi hastaneye götürdük. İdrar kültürü alındı ve Escherichia coli bakterisi saptandı. Bize Cefaks adlı şurubu 10 gün kullandık, 4  ...

💊 DİYABET FİLTRELİ ÖRNEK CEVAP:
 Öncelikle geçmiş olsun, bu bakteri bebeklerin dışkısında normalde bulunan bir mikroptur. Bu nedenle idrar alınırken bulaşmış olabilir, şüphede kaldığınız hastalarda örneğin iğne aracılığıyla direk mes ...

Not: Artık genel verilerin olduğu '06_hf_mega_qa_dataset.csv' dosyasını silebilirsin. Bize sadece bu yeni dosya lazım.
